# World Bank Cost Data Preparation

This notebook prepares the World Bank Pink Sheet cost data for the later classification stage. It reads the raw Excel files directly, extracts the real material names and units, reshapes the annual and monthly price tables, and saves a minimal annual cost dataset with `material`, `cost`, `unit`, and `year`.

In [1]:
# Import the libraries used in this notebook section.
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

In [2]:
# Set up project paths so files can be read and outputs can be organised consistently.
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

raw_dir = project_root / "data" / "raw" / "cost_data" / "world_bank"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

annual_path = raw_dir / "CMO-Historical-Data-Annual.xlsx"
monthly_path = raw_dir / "CMO-Historical-Data-Monthly.xlsx"

annual_output = processed_dir / "world_bank_cost_annual_cleaned.csv"
monthly_output = processed_dir / "world_bank_cost_monthly_cleaned.csv"
classification_output = processed_dir / "cost_for_classification.csv"

print("Annual workbook:", annual_path)
print("Monthly workbook:", monthly_path)
print("Classification-ready output:", classification_output)

Annual workbook: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\raw\cost_data\world_bank\CMO-Historical-Data-Annual.xlsx
Monthly workbook: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\raw\cost_data\world_bank\CMO-Historical-Data-Monthly.xlsx
Classification-ready output: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\processed\cost_for_classification.csv


In [3]:
# Check that the expected input files are available before continuing.
for path in [annual_path, monthly_path]:
    if not path.exists():
        raise FileNotFoundError(f"Could not find workbook at {path}")

## Workbook structure

In [4]:
annual_xls = pd.ExcelFile(annual_path)
monthly_xls = pd.ExcelFile(monthly_path)

print("Annual sheets:", annual_xls.sheet_names)
print("Monthly sheets:", monthly_xls.sheet_names)

Annual sheets: ['AFOSHEET', 'Annual Prices (Nominal)', 'Annual Indices (Nominal)', 'Annual Prices (Real)', 'Annual Indices (Real)', 'Description', 'Definitions', 'Index Weights']
Monthly sheets: ['AFOSHEET', 'Monthly Prices', 'Monthly Indices', 'Description', 'Index Weights']


## Load the relevant price sheets

These row positions are based on inspection of the World Bank workbook structure:
- Annual Prices (Nominal): header row 6, unit row 7, data from row 8
- Monthly Prices: header row 4, unit row 5, data from row 6

In [5]:
ANNUAL_SHEET = "Annual Prices (Nominal)"
MONTHLY_SHEET = "Monthly Prices"

ANNUAL_HEADER_ROW = 6
ANNUAL_UNIT_ROW = 7
ANNUAL_DATA_START = 8

MONTHLY_HEADER_ROW = 4
MONTHLY_UNIT_ROW = 5
MONTHLY_DATA_START = 6

# Load the dataset needed for the next analysis step.
annual_raw = pd.read_excel(annual_path, sheet_name=ANNUAL_SHEET, header=None)
monthly_raw = pd.read_excel(monthly_path, sheet_name=MONTHLY_SHEET, header=None)

# Inspect the data to confirm the structure and values look reasonable.
print("Annual raw shape:", annual_raw.shape)
display(annual_raw.head(12))

print("Monthly raw shape:", monthly_raw.shape)
display(monthly_raw.head(12))

Annual raw shape: (74, 70)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69
0,World Bank Commodity Price Data (The Pink Sheet),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"annual prices, 1960 to present, nominal US dollars",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(annual series are available in nominal and real dollars),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Updated on March 03, 2026",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,"Crude oil, average","Crude oil, Brent","Crude oil, Dubai","Crude oil, WTI","Coal, Australian","Coal, South African","Natural gas, US","Natural gas, Europe","Liquefied natural gas, Japan",Natural gas index,Cocoa,"Coffee, Arabica","Coffee, Robusta","Tea, avg 3 auctions","Tea, Colombo","Tea, Kolkata","Tea, Mombasa",Coconut oil,Groundnuts,Fish meal,Groundnut oil,Palm oil,Palm kernel oil,Soybeans,Soybean oil,Soybean meal,Barley,Maize,Sorghum,"Rice, Thai 5%","Rice, Thai 25%","Rice, Thai A.1","Rice, Viet Namese 5%","Wheat, US SRW","Wheat, US HRW","Banana, Europe","Banana, US",Orange,Beef,Chicken,Lamb **,"Shrimps, Mexican","Sugar, EU","Sugar, US","Sugar, world","Tobacco, US import u.v.","Logs, Cameroon","Logs, Malaysian","Sawnwood, Cameroon","Sawnwood, Malaysian",Plywood,"Cotton, A Index","Rubber, TSR20 **","Rubber, RSS3",Phosphate rock,DAP,TSP,Urea,Potassium chloride,Aluminum,"Iron ore, cfr spot",Copper,Lead,Tin,Nickel,Zinc,Gold,Platinum,Silver
7,NaN,($/bbl),($/bbl),($/bbl),($/bbl),($/mt),($/mt),($/mmbtu),($/mmbtu),($/mmbtu),(2010=100),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/mt),($/cubic meter),($/cubic meter),($/cubic meter),($/cubic meter),(¢/sheet),($/kg),($/kg),($/kg),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/dmtu),($/mt),($/mt),($/mt),($/mt),($/mt),($/troy oz),($/troy oz),($/troy oz)
8,1960,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,…,0.589017,0.923517,0.695946,1.0297,0.930301,1.121401,1.0374,312.333333,…,…,327,224.416667,…,91.833333,223.916667,81.008333,20.153697,44.5,36.575,107.349167,…,…,…,…,57.993333,…,0.142868,0.129983,0.736533,0.30013,…,1.605698,0.122356,0.125847,0.066208,1736.87,…,31.94,…,149.174978,…,0.654542,…,0.780233,13,..,53,42.25,28.5,511.471832,11.42,678.755833,198.85,2196.733333,16

Monthly raw shape: (801, 72)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71
0,World Bank Commodity Price Data (The Pink Sheet),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"monthly prices in nominal US dollars, 1960 to present",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(monthly series are available only in nominal US dollars),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Updated on April 02, 2026",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,"Crude oil, average","Crude oil, Brent","Crude oil, Dubai","Crude oil, WTI","Coal, Australian","Coal, South African **","Natural gas, US","Natural gas, Europe","Liquefied natural gas, Japan",Natural gas index,Cocoa,"Coffee, Arabica","Coffee, Robusta","Tea, avg 3 auctions","Tea, Colombo","Tea, Kolkata","Tea, Mombasa",Coconut oil,Groundnuts,Fish meal,Groundnut oil **,Palm oil,Palm kernel oil,Soybeans,Soybean oil,Soybean meal,Rapeseed oil,Sunflower oil,Barley,Maize,Sorghum,"Rice, Thai 5%","Rice, Thai 25%","Rice, Thai A.1","Rice, Viet Namese 5%","Wheat, US SRW","Wheat, US HRW","Banana, Europe","Banana, US",Orange,Beef **,Chicken **,Lamb **,"Shrimps, Mexican","Sugar, EU","Sugar, US","Sugar, world","Tobacco, US import u.v.","Logs, Cameroon","Logs, Malaysian","Sawnwood, Cameroon","Sawnwood, Malaysian",Plywood,"Cotton, A Index","Rubber, TSR20 **","Rubber, RSS3",Phosphate rock,DAP,TSP,Urea,Potassium chloride **,Aluminum,"Iron ore, cfr spot",Copper,Lead,Tin,Nickel,Zinc,Gold,Platinum,Silver
5,NaN,($/bbl),($/bbl),($/bbl),($/bbl),($/mt),($/mt),($/mmbtu),($/mmbtu),($/mmbtu),(2010=100),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/mt),($/cubic meter),($/cubic meter),($/cubic meter),($/cubic meter),(cents/sheet),($/kg),($/kg),($/kg),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/dmtu),($/mt),($/mt),($/mt),($/mt),($/mt),($/troy oz),($/troy oz),($/troy oz)
6,1960M01,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,…,0.634,0.9409,0.696864,1.0297,0.930301,1.121401,1.0374,390,…,…,334,233,…,94,204,91.9,…,…,20.3648,45,39,104.45,…,…,…,…,59.89,…,0.14308,0.1151,0.7055,0.297369,…,1.433003,0.122356,0.116845,0.0666,1736.87,…,31.94,…,149.174978,…,0.6486,…,0.8223,13,...,53,42.25,28.5,511.471832,11.42,715.4,206.1,2180.4,1631,260.8,35.27,83.5,0.9137
7,1960M02,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,…,0.608,0.9469,0.688707,1.0297,0.930301,1.121401,1.0374,379,…,…,341,229,…,91,201,86.7,…,…,20.410464,44,39,103.54,…,…,…,…,60.99,…,0.14308,0.1092,0.7121,0.297425,…,1.499142,0.122356,0.119049,0.0679,1736.87,…,31.94,…,149.174978,…,0.6453,…,0.8289,13,...,53,42.25,28.5,511.471832,11.42,728.19,203.7,2180.4,1631,244.9,35.27,83.5,0.9137
8,1960M03,1.63,1.63,1.63,…,…,…,0.14,0.404774,…,…,0.5789,0.9281,0.688707,1.0297,0.930301,1.121401,1.0374,361,…,…,338,225,…,92,2

## Parse the price sheets

In [6]:
def parse_world_bank_price_sheet(df, header_row_idx, unit_row_idx, data_start_idx):
    # Handle missing values so later transformations and models do not fail.
    header_row = df.iloc[header_row_idx].fillna("")
    # Filter the data to keep the records relevant for this step.
    unit_row = df.iloc[unit_row_idx].fillna("")
    data = df.iloc[data_start_idx:].copy()

    data = data.dropna(axis=0, how="all").dropna(axis=1, how="all")
    if data.empty:
        raise ValueError("No usable data rows found after applying the configured start row.")

    column_names = ["period"]
    for idx, value in enumerate(header_row.iloc[1:], start=1):
        text = str(value).strip()
        if text == "" or text.lower() == "nan":
            text = f"unnamed_{idx}"
        column_names.append(text)

    data.columns = column_names

    unit_lookup = {}
    for col, unit_value in zip(column_names[1:], unit_row.iloc[1:]):
        unit_text = str(unit_value).strip()
        unit_lookup[col] = None if unit_text == "" or unit_text.lower() == "nan" else unit_text

    # Convert values into analysis-friendly numeric, date, or text formats.
    data["period"] = data["period"].astype(str).str.strip()
    data = data[data["period"] != ""]
    data = data[~data["period"].str.lower().isin(["nan"])]

    long_df = data.melt(id_vars=["period"], var_name="material", value_name="cost")
    long_df["material"] = long_df["material"].astype(str).str.strip()
    long_df["material_match_key"] = long_df["material"].str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
    long_df["cost"] = pd.to_numeric(long_df["cost"], errors="coerce")
    long_df = long_df.dropna(subset=["cost"]).copy()
    long_df["unit"] = long_df["material"].map(unit_lookup)

    return long_df.reset_index(drop=True)

annual_long = parse_world_bank_price_sheet(
    annual_raw,
    header_row_idx=ANNUAL_HEADER_ROW,
    unit_row_idx=ANNUAL_UNIT_ROW,
    data_start_idx=ANNUAL_DATA_START,
)

monthly_long = parse_world_bank_price_sheet(
    monthly_raw,
    header_row_idx=MONTHLY_HEADER_ROW,
    unit_row_idx=MONTHLY_UNIT_ROW,
    data_start_idx=MONTHLY_DATA_START,
)

# Inspect the data to confirm the structure and values look reasonable.
print("Annual tidy shape:", annual_long.shape)
display(annual_long.head(20))

print("Monthly tidy shape:", monthly_long.shape)
display(monthly_long.head(20))

Annual tidy shape: (4099, 4)


,period,material,cost,unit
0,1960,"Crude oil, average",1.630000,($/bbl)
1,1961,"Crude oil, average",1.570000,($/bbl)
2,1962,"Crude oil, average",1.520000,($/bbl)
3,1963,"Crude oil, average",1.500000,($/bbl)
4,1964,"Crude oil, average",1.450000,($/bbl)
5,1965,"Crude oil, average",1.420000,($/bbl)
6,1966,"Crude oil, average",1.360000,($/bbl)
7,1967,"Crude oil, average",1.330000,($/bbl)
8,1968,"Crude oil, average",1.320000,($/bbl)
9,1969,"Crude oil, average",1.270000,($/bbl)


Monthly tidy shape: (50031, 4)


,period,material,cost,unit
0,1960M01,"Crude oil, average",1.63,($/bbl)
1,1960M02,"Crude oil, average",1.63,($/bbl)
2,1960M03,"Crude oil, average",1.63,($/bbl)
3,1960M04,"Crude oil, average",1.63,($/bbl)
4,1960M05,"Crude oil, average",1.63,($/bbl)
5,1960M06,"Crude oil, average",1.63,($/bbl)
6,1960M07,"Crude oil, average",1.63,($/bbl)
7,1960M08,"Crude oil, average",1.63,($/bbl)
8,1960M09,"Crude oil, average",1.63,($/bbl)
9,1960M10,"Crude oil, average",1.63,($/bbl)


## Build minimal classification-ready cost table

In [7]:
cost_for_classification = annual_long.copy()

# Convert values into analysis-friendly numeric, date, or text formats.
cost_for_classification["year"] = pd.to_numeric(cost_for_classification["period"], errors="coerce")
# Handle missing values so later transformations and models do not fail.
cost_for_classification = cost_for_classification.dropna(subset=["year"]).copy()
# Filter the data to keep the records relevant for this step.
cost_for_classification["year"] = cost_for_classification["year"].astype(int)

latest_year = cost_for_classification["year"].max()
cost_for_classification = cost_for_classification[cost_for_classification["year"] == latest_year].copy()

cost_for_classification = cost_for_classification[["material", "material_match_key", "cost", "unit", "year"]]
# Summarise the data so patterns can be compared more easily.
cost_for_classification = cost_for_classification.drop_duplicates(subset=["material", "unit"]).sort_values("cost").reset_index(drop=True)

print("Latest annual year used:", latest_year)
# Inspect the data to confirm the structure and values look reasonable.
print("Classification-ready shape:", cost_for_classification.shape)
display(cost_for_classification.head(20))

Latest annual year used: 2025
Classification-ready shape: (66, 4)


,material,cost,unit,year
0,"Sugar, EU",0.368912,($/kg),2025
1,"Sugar, world",0.371717,($/kg),2025
2,"Sugar, US",0.792157,($/kg),2025
3,"Banana, US",1.095306,($/kg),2025
4,"Banana, Europe",1.109238,($/kg),2025
5,Orange,1.417542,($/kg),2025
6,"Cotton, A Index",1.706964,($/kg),2025
7,Chicken,1.713614,($/kg),2025
8,"Rubber, TSR20 **",1.769217,($/kg),2025
9,"Tea, Mombasa",2.146458,($/kg),2025


## Save outputs

In [8]:
# Save the processed output so later notebooks or report sections can reuse it.
annual_long.to_csv(annual_output, index=False)
monthly_long.to_csv(monthly_output, index=False)
cost_for_classification.to_csv(classification_output, index=False)

print(f"Saved annual cleaned prices to: {annual_output}")
print(f"Saved monthly cleaned prices to: {monthly_output}")
print(f"Saved classification-ready cost data to: {classification_output}")

Saved annual cleaned prices to: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\processed\world_bank_cost_annual_cleaned.csv
Saved monthly cleaned prices to: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\processed\world_bank_cost_monthly_cleaned.csv
Saved classification-ready cost data to: C:\Users\spinn\OneDrive\Documents\NCI AI Masters\Data analytics for Artificial intelligence\Project\data\processed\cost_for_classification.csv
